## TECH CHALLENGE - FASE 4

## Análise Multimodal para Detecção de Riscos em Saúde da Mulher

Este notebook apresenta a implementação inicial de um pipeline multimodal capaz de analisar dados de vídeo e áudio com o objetivo de identificar possíveis sinais de risco clínico e psicológico.

A solução é uma evolução do assistente médico desenvolvido na fase anterior, incorporando análise de comportamento (vídeo) e linguagem (áudio), permitindo maior capacidade de detecção precoce de situações críticas.

## Objetivo

Desenvolver um pipeline multimodal que:

- Analise vídeos para identificar padrões comportamentais e sinais de desconforto
- Processe áudios de consultas para detectar sinais emocionais e psicológicos
- Combine os resultados das diferentes modalidades
- Gere um alerta baseado no nível de risco identificado

Este sistema atua como suporte à triagem clínica, não realizando diagnósticos médicos.

## Importação de Bibliotecas e Módulos

Nesta etapa, são importadas as funções desenvolvidas no projeto para análise de vídeo, áudio e fusão multimodal.

In [1]:
import sys
sys.path.append("..")

In [2]:
from src.multimodal.video_processor import analyze_video
from src.multimodal.audio_processor import analyze_audio
from src.multimodal.multimodal_fusion import calculate_multimodal_risk
from src.multimodal.alert_generator import generate_alert

## Configuração do Ambiente

Configuração de caminhos e preparação do ambiente para execução das análises.

In [3]:
from pathlib import Path

print(Path.cwd())

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks


In [4]:
video_result = analyze_video("../data/multimodal/video/sample.mp4")
audio_result = analyze_audio("../data/multimodal/audio/sample.wav")

risk_result = calculate_multimodal_risk(video_result, audio_result)
alert = generate_alert(risk_result)

video_result, audio_result, risk_result, alert

({'modality': 'video',
  'risk_score': 0.55,
  'risk_level': 'medio',
  'flags': ['movimento corporal identificado',
   'análise visual inicial executada'],
  'metadata': {'video_path': '..\\data\\multimodal\\video\\sample.mp4',
   'fps': 0,
   'frame_count': 0,
   'duration_seconds': 0,
   'status': 'mock_video_file'}},
 {'modality': 'audio',
  'risk_score': 0.25,
  'risk_level': 'baixo',
  'flags': [],
  'detected_categories': {},
  'voice_features': {'duration_seconds': None,
   'rms_energy': None,
   'voice_intensity': 'indisponível'},
  'transcription': 'Não foi possível transcrever o áudio.',
  'interpretation': 'Não foram identificados sinais emocionais relevantes na transcrição. A recomendação é manter acompanhamento regular.',
  'recommendation': 'Recomenda-se acompanhamento regular.'},
 {'final_score': 0.37,
  'risk_level': 'baixo',
  'alert': False,
  'evidences': ['movimento corporal identificado',
   'análise visual inicial executada'],
  'video_score': 0.55,
  'audio_scor

## Carregamento de Dados de Entrada

Nesta fase inicial, são utilizados dados simulados (mock) para representar arquivos de vídeo e áudio.
Em etapas futuras, estes dados serão substituídos por arquivos reais e integrados a serviços como Azure Speech e modelos de visão computacional.

## Análise de Vídeo

O módulo de análise de vídeo tem como objetivo identificar padrões visuais que possam indicar:

- Desconforto físico
- Movimentos anormais
- Postura corporal inadequada

Nesta etapa inicial, a análise é simulada e será futuramente substituída por modelos de visão computacional (YOLOv8).

## Análise de Áudio

O módulo de análise de áudio processa informações da fala da paciente, buscando identificar:

- Sinais de ansiedade
- Indícios de medo
- Relatos de sintomas relevantes

A análise utiliza transcrição simulada, sendo posteriormente integrada com serviços como Azure Speech-to-Text.

In [5]:
import sys
sys.path.append("..")

from src.workflows.langgraph_flow import build_medical_assistant_graph
from src.llm.ollama_client import get_llm
from src.rag.vector_store import load_vector_store
from src.rag.retriever import get_retriever
from src.rag.retriever import retrieve_context

In [6]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks\..\src\llm\ollama_client.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


Base vetorial carregada de: ../data/vector_store


## Execução do Fluxo Multimodal com LangGraph

Nesta etapa, o pipeline multimodal é executado por meio do LangGraph, mantendo a arquitetura evolutiva desenvolvida na fase anterior.

O fluxo recebe como entrada uma pergunta clínica, um identificador opcional de paciente, um arquivo de vídeo e um arquivo de áudio. Em seguida, o sistema executa os nós de guardrails, recuperação de contexto via RAG, análise de vídeo, análise de áudio, fusão multimodal, geração de resposta e registro da interação.

Nesta execução inicial, os dados de áudio e vídeo ainda são simulados, permitindo validar a integração entre os módulos antes da substituição por modelos reais, como YOLOv8 para vídeo, Azure Speech-to-Text para transcrição e GPT para interpretação textual especializada.

In [7]:
state = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/sample.wav"
}

result = app.invoke(state)
result

{'question': 'Avalie possíveis sinais de risco no atendimento da paciente.',
 'risk_level': 'low',
 'action': 'allow',
 'reason': 'Pergunta informativa.',
 'docs': [Document(id='b0801d89-44f4-4f6c-8a1e-b74b7a03c78a', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='fe392124-22dc-406d-a1cb-da95babe86e9', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='a29fa53d-b57a-47d2-a544-780df6879c83', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia')],
 'context': 'Recurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia',
 'patient_id': None,
 'patient_context': 'No patien

## Interpretação do Resultado

O resultado demonstra que o fluxo multimodal foi executado com sucesso.

A saída contém:

- A classificação de risco inicial realizada pelos guardrails
- Os documentos recuperados pela base vetorial FAISS
- O contexto estruturado do paciente, quando disponível
- O contexto multimodal gerado a partir das análises de vídeo e áudio
- O score final de risco multimodal
- A resposta gerada pelo assistente médico
- As fontes utilizadas para rastreabilidade
- O status final da execução

Nesta primeira versão, o risco final foi classificado como médio, com evidências simuladas relacionadas a movimento corporal, medo, cansaço e dificuldade para dormir. O alerta crítico não foi acionado, mas o sistema recomenda acompanhamento regular pela equipe de saúde.

Essa etapa confirma que a arquitetura da Fase 3 foi expandida para suportar dados multimodais, mantendo os princípios de segurança, explicabilidade e rastreabilidade.

In [8]:
print(result["answer"])
print(result["multimodal_result"])

 Based on the provided structured patient data, multimodal analysis, and evidence, there are no critical alerts detected in this patient's current assessment. The multimodal score is 0.4, indicating a medium risk level. The patient has shown identified body movement and an initial visual analysis has been executed. Therefore, it is recommended to continue regular follow-up with the healthcare team.

However, it's important to note that Recurrent Adult Acute Lymphoblastic Leukemia (ALL) was mentioned in the medical knowledge retrieved section. This information might be relevant to the patient's condition, but without specific patient data, it is not possible to definitively assess its significance. It would be advisable for the healthcare team to consider this potential diagnosis during further evaluation and follow-up.

In summary, while there are no critical alerts at the moment, the healthcare team should remain vigilant regarding the possibility of Recurrent Adult Acute Lymphoblasti

In [9]:
print(result["multimodal_context"])

MULTIMODAL ANALYSIS:
Video risk score: 0.55
Audio risk score: 0.3
Final multimodal score: 0.4
Risk level: medio
Alert generated: False

Evidence:
- movimento corporal identificado
- análise visual inicial executada

Alert message:
Sem alerta crítico no momento. Recomenda-se acompanhamento regular pela equipe de saúde.


In [10]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app1 = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

Base vetorial carregada de: ../data/vector_store


In [11]:
state = {
    "question": "Avalie possíveis sinais de câncer de mama.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/sample.wav"
}

result1 = app1.invoke(state)
result1

{'question': 'Avalie possíveis sinais de câncer de mama.',
 'risk_level': 'low',
 'action': 'allow',
 'reason': 'Pergunta informativa.',
 'docs': [Document(id='b0801d89-44f4-4f6c-8a1e-b74b7a03c78a', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='fe392124-22dc-406d-a1cb-da95babe86e9', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia'),
  Document(id='a29fa53d-b57a-47d2-a544-780df6879c83', metadata={'source_file': '0000001_1.xml', 'collection': 'CancerGov', 'id': '0000001_1.xml_6'}, page_content='Recurrent Adult Acute Lymphoblastic Leukemia')],
 'context': 'Recurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia\nRecurrent Adult Acute Lymphoblastic Leukemia',
 'patient_id': None,
 'patient_context': 'No patient data was provide

In [13]:
print(result1["answer"])
print(result1["multimodal_result"])

 Based on the provided context, there are no signs or symptoms of breast cancer detected in this case. The multimodal analysis indicates a medium risk level with no critical alert. The identified symptoms include movement observed, initial visual analysis, fear, fatigue, and difficulty sleeping. These symptoms could be associated with various conditions such as stress, anxiety, or other non-cancerous health issues. However, without specific patient data, it is not possible to definitively assess the possibility of breast cancer. It's recommended that the patient follows regular check-ups with their healthcare team for thorough evaluation and appropriate care.
{'final_score': 0.67, 'risk_level': 'medio', 'alert': False, 'evidences': ['movimento corporal identificado', 'análise visual inicial executada', 'medo', 'cansaço', 'dificuldade para dormir'], 'video_score': 0.55, 'audio_score': 0.75}


In [14]:
print(result1["multimodal_context"])

MULTIMODAL ANALYSIS:
Video risk score: 0.55
Audio risk score: 0.75
Final multimodal score: 0.67
Risk level: medio
Alert generated: False

Evidence:
- movimento corporal identificado
- análise visual inicial executada
- medo
- cansaço
- dificuldade para dormir

Alert message:
Sem alerta crítico no momento. Recomenda-se acompanhamento regular pela equipe de saúde.


## Análise do Resultado – Limitações da Abordagem Multimodal

Neste experimento, foi realizada uma pergunta sobre possíveis sinais de câncer de mama, enquanto o pipeline multimodal utilizou dados comportamentais e emocionais simulados (vídeo e áudio).

O sistema respondeu corretamente ao indicar que não é possível identificar câncer de mama com base apenas em sinais comportamentais ou emocionais, reforçando a necessidade de exames clínicos e dados médicos específicos.

A análise multimodal contribuiu apenas com um score de risco geral (médio), baseado em sinais como medo, cansaço e dificuldade para dormir, que são inespecíficos e podem estar associados a diversas condições não relacionadas ao câncer.

Esse resultado evidencia uma limitação importante da abordagem:
dados multimodais como áudio e vídeo são úteis para triagem e identificação de bem-estar psicológico, mas não são suficientes para diagnóstico de doenças específicas como câncer.

O sistema manteve comportamento seguro, evitando conclusões médicas indevidas e recomendando acompanhamento profissional, alinhado aos princípios de segurança definidos na arquitetura.

In [7]:
import speech_recognition as sr

print("OK")

OK


In [8]:
import speech_recognition as sr

def transcribe_audio(audio_path):
    recognizer = sr.Recognizer()

    with sr.AudioFile(audio_path) as source:
        audio = recognizer.record(source)

    text = recognizer.recognize_google(audio, language="pt-BR")

    return text


audio_path = "../data/multimodal/audio/audio1.wav"

transcription = transcribe_audio(audio_path)
print(transcription)

audio_path2 = "../data/multimodal/audio/audio2.wav"

transcription2 = transcribe_audio(audio_path2)
print(transcription2)

audio_path3 = "../data/multimodal/audio/audio3.wav"

transcription3 = transcribe_audio(audio_path3)
print(transcription3)

Oi boa tarde venho apenas para uma consulta de rotina e gostaria de fazer um check-up geral para avaliar como está minha saúde
eu tenho me sentindo muito cansada ultimamente um pouco ansiosa fico um pouco agitada também às vezes eu esqueço das coisas muito rápido
eu tenho me sentido muito mal eu acordo fico com vontade de ficar na cama o dia inteiro eu não consigo dormir fico com uma sensação de angústia medo ao longo do dia não tenho vontade de fazer nada


In [9]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

Base vetorial carregada de: ../data/vector_store


In [10]:
state = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/audio1.wav"
}

In [11]:
result = app.invoke(state)

print(result["audio_result"]["transcription"])
print(result["audio_result"]["flags"])
print(result["multimodal_result"])

Oi boa tarde venho apenas para uma consulta de rotina e gostaria de fazer um check-up geral para avaliar como está minha saúde
[]
{'final_score': 0.43, 'risk_level': 'medio', 'alert': False, 'evidences': ['movimento corporal identificado', 'análise visual inicial executada'], 'video_score': 0.55, 'audio_score': 0.35}


In [17]:
state1 = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/audio2.wav"
}

result1 = app.invoke(state1)

print(result1["audio_result"]["transcription"])
print(result1["audio_result"]["flags"])
print(result1["multimodal_result"])

eu tenho me sentindo muito cansada ultimamente um pouco ansiosa fico um pouco agitada também às vezes eu esqueço das coisas muito rápido
['cansada', 'ansiosa', 'agitada']
{'final_score': 0.67, 'risk_level': 'medio', 'alert': False, 'evidences': ['movimento corporal identificado', 'análise visual inicial executada', 'cansada', 'ansiosa', 'agitada'], 'video_score': 0.55, 'audio_score': 0.75}


In [18]:
state2 = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "patient_id": None,
    "video_path": "../data/multimodal/video/sample.mp4",
    "audio_path": "../data/multimodal/audio/audio3.wav"
}

result2 = app.invoke(state2)

print(result2["audio_result"]["transcription"])
print(result2["audio_result"]["flags"])
print(result2["multimodal_result"])

eu tenho me sentido muito mal eu acordo fico com vontade de ficar na cama o dia inteiro eu não consigo dormir fico com uma sensação de angústia medo ao longo do dia não tenho vontade de fazer nada
['medo', 'angústia', 'não consigo dormir', 'não tenho vontade', 'muito mal']
{'final_score': 0.82, 'risk_level': 'alto', 'alert': True, 'evidences': ['movimento corporal identificado', 'análise visual inicial executada', 'medo', 'angústia', 'não consigo dormir', 'não tenho vontade', 'muito mal'], 'video_score': 0.55, 'audio_score': 1.0}


## Teste com Áudio Real

Nesta etapa, o pipeline foi atualizado para utilizar um arquivo de áudio real. A transcrição automática foi executada com sucesso e integrada ao fluxo multimodal do LangGraph.

O resultado demonstra que a arquitetura já é capaz de substituir dados simulados por dados reais sem alterar o restante do pipeline. Nesta execução, a análise de áudio ainda depende de palavras-chave, o que pode limitar a detecção de risco caso a transcrição não contenha exatamente os termos esperados.

Como próximo passo, o módulo de análise textual será refinado com mais variações linguísticas e, futuramente, poderá ser integrado a modelos de linguagem para interpretação contextual mais robusta.

In [8]:
import speech_recognition as sr

print("SpeechRecognition instalado!")

SpeechRecognition instalado!


In [8]:
from src.multimodal.audio_processor import analyze_audio

audio_result = analyze_audio("../data/multimodal/audio/audio3.wav")

print(audio_result["transcription"])
print(audio_result["detected_categories"])
print(audio_result["voice_features"])
print(audio_result["interpretation"])
print(audio_result["recommendation"])

eu tenho me sentido muito mal eu acordo fico com vontade de ficar na cama o dia inteiro eu não consigo dormir fico com uma sensação de angústia medo ao longo do dia não tenho vontade de fazer nada
{'ansiedade': ['angústia', 'medo'], 'depressao_pos_parto_ou_sofrimento_emocional': ['não tenho vontade', 'muito mal'], 'sinais_de_violencia_ou_medo': ['medo'], 'alteracao_do_sono': ['não consigo dormir']}
{'duration_seconds': 26.91, 'rms_energy': 2269, 'voice_intensity': 'alta'}
A análise identificou categorias associadas a ansiedade, depressao_pos_parto_ou_sofrimento_emocional, sinais_de_violencia_ou_medo, alteracao_do_sono. Os principais indicadores encontrados foram: angústia, medo, não tenho vontade, muito mal, não consigo dormir. Esses sinais não representam diagnóstico, mas podem indicar necessidade de atenção especializada.
Recomenda-se avaliação prioritária pela equipe especializada.


## Interpretação da Análise de Áudio com Dados Reais

Nesta etapa, foi realizada a análise de um áudio real contendo relatos de desconforto emocional e possíveis alterações comportamentais.

A transcrição automática identificou expressões relevantes como “angústia”, “medo”, “não tenho vontade” e “não consigo dormir”, permitindo a classificação em múltiplas categorias emocionais, incluindo:

- ansiedade  
- sofrimento emocional (possível depressão pós-parto)  
- sinais de medo ou vulnerabilidade  
- alteração do sono  

Além da análise textual, foram extraídas características do sinal de áudio, como duração e intensidade vocal. A intensidade foi classificada como **alta**, o que pode indicar maior carga emocional durante a fala.

Com base na combinação desses fatores, o sistema atribuiu um nível de risco elevado, recomendando avaliação prioritária por equipe especializada.

É importante destacar que os resultados não representam um diagnóstico clínico, mas sim um mecanismo de triagem inicial baseado em padrões linguísticos e comportamentais. A abordagem multimodal permite identificar sinais precoces de sofrimento psicológico, contribuindo para intervenções mais rápidas e direcionadas.

In [12]:
import os
print(os.getcwd())

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks


In [13]:
import os

print(os.path.exists(".env"))

True


In [14]:
import sys
!{sys.executable} -m pip install boto3 python-dotenv


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Integração com AWS S3 – Teste de Conectividade

Na célula abaixo foi realizada a validação da integração do sistema com o serviço de armazenamento em nuvem Amazon S3, garantindo que o pipeline seja capaz de:

- Realizar upload de arquivos
- Realizar download de arquivos
- Remover arquivos após processamento

Essa etapa é essencial para suportar a arquitetura multimodal do projeto, permitindo o armazenamento seguro de áudios e vídeos.

---

### Configuração

Foram utilizadas as seguintes tecnologias:

- AWS S3 para armazenamento em nuvem
- boto3 como SDK Python para integração com a AWS
- python-dotenv para gerenciamento de credenciais

As credenciais foram armazenadas em um arquivo `.env`:

```env
AWS_ACCESS_KEY_ID=***
AWS_SECRET_ACCESS_KEY=***
AWS_REGION=us-east-1
AWS_S3_BUCKET=nome-do-bucket

In [12]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

s3 = boto3.client("s3", region_name=region)

# cria pasta temp
os.makedirs("temp", exist_ok=True)

# arquivo local
with open("temp/teste.txt", "w") as f:
    f.write("Teste S3 FIAP")

# upload
s3.upload_file("temp/teste.txt", bucket, "temp/teste.txt")
print("Upload OK")

# download
s3.download_file(bucket, "temp/teste.txt", "temp/teste_baixado.txt")
print("Download OK")

# leitura
with open("temp/teste_baixado.txt") as f:
    print("Conteúdo:", f.read())

# delete
s3.delete_object(Bucket=bucket, Key="temp/teste.txt")
print("Delete OK")

Upload OK
Download OK
Conteúdo: Teste S3 FIAP
Delete OK


## Upload de áudio no AWS S3

Áudio real importado no bucket criado no AWS S3.

In [20]:
audio_local = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\audio2.wav"

s3.upload_file(
    audio_local,
    bucket,
    "inputs/audio/audio_real.wav"
)

print("Upload realizado com sucesso")

Upload realizado com sucesso


In [22]:
llm = get_llm(model_name="mistral", temperature=0.2)

vector_store = load_vector_store("../data/vector_store")
retriever = get_retriever(vector_store, k=3)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

Base vetorial carregada de: ../data/vector_store


In [23]:
state = {
    "question": "Avalie possíveis sinais de risco no atendimento da paciente.",
    "audio_s3_key": "inputs/audio/audio_real.wav",
    "video_s3_key": None,
    "patient_id": None
}

result = app.invoke(state)

print(result["audio_result"]["transcription"])
print(result["audio_result"]["flags"])
print(result["multimodal_result"])

eu tenho me sentindo muito cansada ultimamente um pouco ansiosa fico um pouco agitada também às vezes eu esqueço das coisas muito rápido
['ansiosa', 'agitada', 'cansada']
{'final_score': 0.39, 'risk_level': 'baixo', 'alert': False, 'evidences': ['ansiosa', 'agitada', 'cansada'], 'video_score': 0, 'audio_score': 0.65}


## Interpretação do Score Multimodal

### Contexto do Teste

Foi realizado um teste com entrada multimodal contendo:

- Áudio real da paciente (processado via S3)
- Ausência de vídeo (`video_s3_key = None`)
- Pergunta neutra focada em triagem clínica

---

### Resultado Obtido

```json
{
  "final_score": 0.39,
  "risk_level": "baixo",
  "alert": false,
  "evidences": ["ansiosa", "agitada", "cansada"],
  "video_score": 0,
  "audio_score": 0.65
}



### Observações

Apesar do audio_score indicar um nível moderado de risco (0.65), o final_score foi reduzido para 0.39 devido à ausência de dados de vídeo.

Isso evidencia que:

- O sistema considera múltiplas fontes de evidência
- A ausência de uma modalidade pode impactar o resultado final
- A análise multimodal é mais robusta quando ambas as entradas estão disponíveis

Limitações:
- A ausência de vídeo reduz a capacidade de avaliação comportamental
- O score final pode subestimar o risco quando apenas uma modalidade é utilizada
- O sistema não realiza diagnóstico, apenas apoio à triagem

In [24]:
import os
import boto3
from dotenv import load_dotenv

load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

s3 = boto3.client("s3", region_name=region)

# caminho do arquivo no S3 (igual você usou no upload)
s3_key = "inputs/audio/audio_real.wav"

s3.delete_object(Bucket=bucket, Key=s3_key)

print("Arquivo removido do S3 com sucesso")

Arquivo removido do S3 com sucesso


## Etapa de Análise de Vídeo (Multimodal)

### Objetivo

Implementar a análise de vídeo como parte do pipeline multimodal, permitindo a extração de informações visuais que possam complementar a avaliação de risco baseada em áudio e contexto clínico.

Esta etapa tem como finalidade incorporar sinais comportamentais e contextuais, contribuindo para uma análise mais completa, sem caráter diagnóstico.

---

### Tecnologias Utilizadas

- OpenCV: leitura e manipulação de vídeo
- YOLOv8 (Ultralytics): detecção de objetos em frames
- Python: implementação da lógica de análise

---

### Fluxo de Processamento

```text
Vídeo (.mp4)
→ leitura com OpenCV
→ extração de frames
→ detecção com YOLOv8
→ identificação de elementos relevantes (ex: pessoa)
→ geração de métricas e sinais
→ output estruturado

In [16]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import cv2

In [29]:
import os

video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"

print(os.path.exists(video_path))

True


In [30]:
import cv2
import os

video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"
print("Existe?", os.path.exists(video_path))
print("Tamanho bytes:", os.path.getsize(video_path))

cap = cv2.VideoCapture(video_path)

print("Abriu?", cap.isOpened())
print("Frames informados:", cap.get(cv2.CAP_PROP_FRAME_COUNT))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
print("Largura:", cap.get(cv2.CAP_PROP_FRAME_WIDTH))
print("Altura:", cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

ret, frame = cap.read()

print("Conseguiu ler primeiro frame?", ret)
print("Frame:", None if frame is None else frame.shape)

cap.release()

Existe? True
Tamanho bytes: 38292439
Abriu? True
Frames informados: 3326.0
FPS: 30.0
Largura: 1280.0
Altura: 720.0
Conseguiu ler primeiro frame? True
Frame: (720, 1280, 3)


### Validação da Leitura de Vídeo com OpenCV

Validar a capacidade do sistema de:

- localizar o arquivo de vídeo
- abrir o vídeo corretamente
- acessar metadados do conteúdo
- ler frames reais para processamento posterior

Essa etapa é necessária antes da integração com o modelo YOLOv8 e com o pipeline multimodal.


In [31]:
!pip install ultralytics


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
import cv2
from ultralytics import YOLO

# carrega modelo
model = YOLO("yolov8n.pt")

video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

if ret:
    results = model(frame)

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            label = model.names[cls]

            print("Detectado:", label)

cap.release()

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\keity\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

0: 384x640 4 persons, 319.2ms
Speed: 26.3ms preprocess, 319.2ms inference, 49.4ms postprocess per image at shape (1, 3, 384, 640)
Detectado: person
Detectado: person
Detectado: person
Detectado: person


## Teste Inicial com YOLOv8 em Frame de Vídeo

### Objetivo

Validar o uso do modelo YOLOv8 para detecção de objetos em frames extraídos do vídeo real utilizado no projeto.

Nesta etapa, o objetivo foi verificar se o modelo consegue identificar elementos visuais relevantes no vídeo antes da integração completa com o pipeline multimodal.

---

### Procedimento

Foi carregado o modelo pré-treinado `yolov8n.pt` e extraído o primeiro frame do vídeo. Em seguida, o frame foi enviado ao modelo YOLO para detecção.

---

### Resultado Obtido

O modelo identificou múltiplas ocorrências da classe:

```text
person

In [34]:
!pip install deepface

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 15.8 MB/s  0:00:00
   ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
   ---------------------------------------- 1.9/1.9 MB 22.0 MB/s  0:00:00
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
    --------------------------------------- 5.0/351.2 MB 24.4 MB/s eta 0:00:15
   - -------------------------------------- 10.5/351.2 MB 25.6 MB/s eta 0:00:14
   - -------------------------------------- 14.7/351.2 MB 23.3 MB/s eta 0:00:15
   -- ------------------------------------- 19.9/351.2 MB 23.3 MB/s eta 0:00:15
   -- ------------------------------------- 25.4/351.2 MB 23.9 MB/s eta 0:00:14
   --- ------------------------------------ 30.1/351.2 MB 23.7 MB/s eta 0:00:14
   --- ------------------------------------ 34.6/351.2 MB 23.5 MB/s eta 0:00:14
   ---- ----------------------------------- 39.1/351.2 MB 23.3 MB/s eta 0:00:14
   -

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\keity\\OneDrive\\Documentos\\GitHub\\postech\\tech-challenge-ia-diagnostico\\tech-challenge-ia-diagnostico\\venv\\Lib\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\credentials\\call\\gcp_service_account_identity\\gcp_service_account_identity_credentials.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from deepface import DeepFace
import cv2

video_path = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\Unlocking Facial Recognition.mp4"

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

if ret:
    result = DeepFace.analyze(
        frame,
        actions=["emotion"],
        enforce_detection=False
    )

    print(result)

ModuleNotFoundError: No module named 'deepface'